# Task 1: Real-Time Mask Detection using CNN and Webcam

Jupyter Notebook mein chalao (Google Colab mein nahi), `tfenv` environment ke saath.

Pehle ek baar Anaconda Prompt mein:
```
conda create -n tfenv python=3.9
conda activate tfenv
pip install tensorflow
pip install opencv-python scikit-learn
pip install notebook
jupyter notebook
```

## Step 2: Dataset banao (webcam se)
`s` = photo save, `q` = quit.
Pehli baar `class_name = "with_mask"` rakho (mask pehen kar 50 photos), phir `"without_mask"` kar ke dobara run karo.

In [ ]:
import cv2
import os

# Define classes
classes = ["with_mask", "without_mask"]

# Create dataset folders
for cls in classes:
    os.makedirs(f"dataset/{cls}", exist_ok=True)

# Face detector (OpenCV ke saath built-in aata hai)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Choose class manually in notebook
class_name = "with_mask"      # <-- dusri baar "without_mask" likho
TARGET = 50                   # har class ki kitni images

# Pehle se save images ke baad se count shuru karo
count = len(os.listdir(f"dataset/{class_name}"))

cap = cv2.VideoCapture(0)

while count < TARGET:
    ret, frame = cap.read()
    if not ret:
        break

    show = frame.copy()
    cv2.putText(show, f"{class_name}: {count}/{TARGET}  (s=save, q=quit)", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.imshow("Capture - Press 's' to Save, 'q' to Quit", show)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        # Sirf face crop save karo (testing mein bhi face crop hi use hota hai)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) == 0:
            print("Face detect nahi hua, dobara 's' dabao.")
            continue
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        face_img = frame[y:y+h, x:x+w]
        path = f"dataset/{class_name}/{class_name}_{count}.jpg"
        cv2.imwrite(path, face_img)
        print(f"Saved: {path}")
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
print(f"{class_name}: total {count} images")

## Step 3: CNN train karo (80% train / 20% test)

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Load and preprocess dataset
data = []
labels = []
categories = ["with_mask", "without_mask"]

for label, category in enumerate(categories):
    folder_path = os.path.join("dataset", category)
    for filename in os.listdir(folder_path):
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (64, 64))
            data.append(img)
            labels.append(label)

print("Total images:", len(data))

X = np.array(data) / 255.0
y = to_categorical(labels)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", len(X_train), "| Test:", len(X_test))

# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')  # 2 classes: with_mask, without_mask
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

# Test accuracy
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc*100:.2f}%")

# Save the trained model
model.save("mask_detector.keras")
print("Model trained and saved as 'mask_detector.keras'")

## Step 4: Live webcam par test
Webcam khulega aur batayega Mask ya No Mask. `q` dabao to quit.

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model("mask_detector.keras")
labels = ["Mask", "No Mask"]

# Load Haar Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("Haar cascade file load nahi hui.")

# Start webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Webcam access nahi ho raha.")

print("Starting real-time mask detection. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]
        face = cv2.resize(face_img, (64, 64))
        face = face / 255.0
        face = np.reshape(face, (1, 64, 64, 3))

        pred = model.predict(face, verbose=0)[0]
        class_idx = np.argmax(pred)
        confidence = pred[class_idx]
        label = f"{labels[class_idx]}: {confidence*100:.2f}%"

        # Draw rectangle and label
        color = (0, 255, 0) if class_idx == 0 else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Mask Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)